In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ============================================================
# SETTINGS
# ============================================================

ROOT = Path(".").resolve()

FEATURE_SETS = [30, 20, 12]

CALIBRATION = "uncalibrated"

# ------------------------------------------------------------
# COLORS
# ------------------------------------------------------------

# All supervised models shown in gray
BACKGROUND_COLOR = "#BDBDBD"

# Unsupervised comparison
RAW_COLOR = "#E67E22"      # without standardization
STD_COLOR = "#7B4AB3"      # with standardization


# ============================================================
# LOAD DATA
# ============================================================

def load_supervised_data(root, transition, standardized=False):
    """
    Load supervised curve data.

    Structure:
        SUPERVISED/
            AC/
            BCb/

    Files:
        AC_curve_mean.csv
        AC_STD_curve_mean.csv

    Returns:
        mean_df
    """

    folder = root / "SUPERVISED" / transition

    suffix = "_STD" if standardized else ""

    mean_path = folder / f"{transition}{suffix}_curve_mean.csv"

    if not mean_path.exists():
        raise FileNotFoundError(mean_path)

    mean_df = pd.read_csv(mean_path)

    return mean_df


def load_unsupervised_data(root, transition, standardized=False):
    """
    Load unsupervised curve data.

    Structure:
        UNSUPERVISED/
            AC/
            BCb/

    Files:
        AC_UNS_curve_mean.csv
        AC_UNS_STD_curve_mean.csv

    Returns:
        mean_df
    """

    folder = root / "UNSUPERVISED" / transition

    if standardized:
        filename = f"{transition}_UNS_STD_curve_mean.csv"
    else:
        filename = f"{transition}_UNS_curve_mean.csv"

    mean_path = folder / filename

    if not mean_path.exists():
        raise FileNotFoundError(mean_path)

    mean_df = pd.read_csv(mean_path)

    return mean_df


# ============================================================
# FILTER SUPERVISED CURVE
# ============================================================

def filter_supervised_curve(
    df,
    feature_set,
    model,
    x_column,
):
    """
    Select one supervised model and feature set.

    Supervised data may contain multiple calibration types.
    We use only 'uncalibrated'.
    """

    result = df[
        (df["feature_set"] == feature_set)
        & (df["model"] == model)
    ].copy()

    if "calibration" in result.columns:
        result = result[
            result["calibration"] == CALIBRATION
        ].copy()

    return result.sort_values(x_column)


# ============================================================
# FILTER UNSUPERVISED CURVE
# ============================================================

def filter_unsupervised_curve(
    df,
    feature_set,
    model,
    x_column,
):
    """
    Select one unsupervised model and feature set.

    If multiple strides are available, use the smallest
    available stride.
    """

    result = df[
        (df["feature_set"] == feature_set)
        & (df["model"] == model)
    ].copy()

    if result.empty:
        return result

    # --------------------------------------------------------
    # Unsupervised models were calculated for different strides.
    # Use the smallest available stride.
    # --------------------------------------------------------

    if "stride" in result.columns:

        strides = pd.to_numeric(
            result["stride"],
            errors="coerce"
        ).dropna()

        if not strides.empty:

            min_stride = strides.min()

            result = result[
                result["stride"] == min_stride
            ].copy()

    return result.sort_values(x_column)


# ============================================================
# GET UNSUPERVISED MODELS
# ============================================================

def get_unsupervised_models(
    raw_df,
    std_df,
):
    """
    Get all unsupervised models present in either
    standardized or non-standardized data.
    """

    models = set()

    if "model" in raw_df.columns:
        models.update(
            raw_df["model"]
            .dropna()
            .unique()
        )

    if "model" in std_df.columns:
        models.update(
            std_df["model"]
            .dropna()
            .unique()
        )

    return sorted(models)


# ============================================================
# PLOT SUPERVISED BACKGROUND
# ============================================================

def plot_supervised_background(
    ax,
    supervised_raw,
    supervised_std,
    feature_set,
    x_column,
):
    """
    Plot all supervised models in gray.

    They are only the background/reference.
    """

    # ========================================================
    # SUPERVISED — WITHOUT STANDARDIZATION
    # ========================================================

    models_raw = (
        supervised_raw["model"]
        .dropna()
        .unique()
    )

    for model in models_raw:

        curve = filter_supervised_curve(
            supervised_raw,
            feature_set,
            model,
            x_column,
        )

        if curve.empty:
            continue

        ax.errorbar(
            curve[x_column],
            curve["mean"],
            yerr=curve["mean_err"],
            fmt="o-",
            color=BACKGROUND_COLOR,
            ecolor=BACKGROUND_COLOR,
            markersize=2,
            linewidth=0.9,
            capsize=1.5,
            elinewidth=0.6,
            alpha=0.45,
            zorder=1,
        )

    # ========================================================
    # SUPERVISED — WITH STANDARDIZATION
    # ========================================================

    models_std = (
        supervised_std["model"]
        .dropna()
        .unique()
    )

    for model in models_std:

        curve = filter_supervised_curve(
            supervised_std,
            feature_set,
            model,
            x_column,
        )

        if curve.empty:
            continue

        ax.errorbar(
            curve[x_column],
            curve["mean"],
            yerr=curve["mean_err"],
            fmt="o-",
            color=BACKGROUND_COLOR,
            ecolor=BACKGROUND_COLOR,
            markersize=2,
            linewidth=0.9,
            capsize=1.5,
            elinewidth=0.6,
            alpha=0.25,
            zorder=1,
        )


# ============================================================
# PLOT ONE UNSUPERVISED COMPARISON PANEL
# ============================================================

def plot_unsupervised_panel(
    ax,
    supervised_raw,
    supervised_std,
    unsupervised_raw,
    unsupervised_std,
    feature_set,
    model,
    x_column,
):
    """
    One panel:

        gray      = all supervised models
        orange    = selected unsupervised model, raw
        purple    = selected unsupervised model, STD
    """

    # ========================================================
    # SUPERVISED BACKGROUND
    # ========================================================

    plot_supervised_background(
        ax=ax,
        supervised_raw=supervised_raw,
        supervised_std=supervised_std,
        feature_set=feature_set,
        x_column=x_column,
    )

    # ========================================================
    # UNSUPERVISED — WITHOUT STANDARDIZATION
    # ========================================================

    raw_curve = filter_unsupervised_curve(
        unsupervised_raw,
        feature_set,
        model,
        x_column,
    )

    if not raw_curve.empty:

        ax.errorbar(
            raw_curve[x_column],
            raw_curve["mean"],
            yerr=raw_curve["mean_err"],
            fmt="o-",
            color=RAW_COLOR,
            ecolor=RAW_COLOR,
            markersize=3.5,
            linewidth=2.0,
            capsize=2.5,
            elinewidth=0.9,
            label="bez standaryzacji",
            zorder=4,
        )

    # ========================================================
    # UNSUPERVISED — WITH STANDARDIZATION
    # ========================================================

    std_curve = filter_unsupervised_curve(
        unsupervised_std,
        feature_set,
        model,
        x_column,
    )

    if not std_curve.empty:

        ax.errorbar(
            std_curve[x_column],
            std_curve["mean"],
            yerr=std_curve["mean_err"],
            fmt="o-",
            color=STD_COLOR,
            ecolor=STD_COLOR,
            markersize=3.5,
            linewidth=2.0,
            capsize=2.5,
            elinewidth=0.9,
            label="ze standaryzacją",
            zorder=5,
        )

    # ========================================================
    # p = 0.5
    # ========================================================

    ax.axhline(
        0.5,
        color="0.65",
        linestyle=":",
        linewidth=0.8,
        zorder=0,
    )

    # ========================================================
    # PANEL TITLE
    # ========================================================

    ax.set_title(
        f"{feature_set} cech",
        fontsize=13,
        fontweight="bold",
        pad=10,
    )

    # ========================================================
    # AXES
    # ========================================================

    ax.set_xlabel(
        r"$\Delta$" if x_column == "Delta" else r"$K_0$",
        fontsize=11,
    )

    ax.set_ylim(
        -0.05,
        1.05,
    )

    ax.grid(
        alpha=0.25,
        linewidth=0.6,
    )

    ax.tick_params(
        axis="both",
        labelsize=9,
    )


# ============================================================
# CREATE ONE FIGURE FOR ONE MODEL
# ============================================================

def make_unsupervised_figure(
    transition,
    model,
    supervised_raw,
    supervised_std,
    unsupervised_raw,
    unsupervised_std,
    x_column,
):
    """
    Create one figure for one unsupervised model.

    Three panels:
        30 features
        20 features
        12 features
    """

    fig, axes = plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(13.5, 4.5),
        sharey=True,
    )

    # ========================================================
    # PANELS
    # ========================================================

    for ax, feature_set in zip(
        axes,
        FEATURE_SETS,
    ):

        plot_unsupervised_panel(
            ax=ax,
            supervised_raw=supervised_raw,
            supervised_std=supervised_std,
            unsupervised_raw=unsupervised_raw,
            unsupervised_std=unsupervised_std,
            feature_set=feature_set,
            model=model,
            x_column=x_column,
        )

    # ========================================================
    # Y LABEL
    # ========================================================

    axes[0].set_ylabel(
        r"$\langle \Pr(d_j(\Delta)\in C_b)\rangle$",
        fontsize=11,
    )

    # ========================================================
    # LEGEND
    # ========================================================

    handles, labels = axes[0].get_legend_handles_labels()

    if handles:

        fig.legend(
            handles,
            labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.025),
            ncol=2,
            frameon=False,
            fontsize=9,
        )

    # ========================================================
    # TITLE
    # ========================================================

    fig.suptitle(
        f"{transition} — {model}",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )

    # ========================================================
    # LAYOUT
    # ========================================================

    plt.tight_layout(
        rect=[
            0.0,
            0.08,
            1.0,
            0.96,
        ]
    )

    # ========================================================
    # SHOW ONLY
    # ========================================================

    

    return fig, axes


# ============================================================
# CREATE ALL FIGURES FOR ONE TRANSITION
# ============================================================

def make_all_unsupervised_figures(
    transition,
    pdf,
):
    """
    Create all unsupervised comparison figures
    for one transition.

    Example:
        AC — KMeans
        AC — DBSCAN
        AC — Birch
        ...

    or:

        BCb — KMeans
        BCb — DBSCAN
        BCb — Birch
        ...
    """

    # ========================================================
    # X AXIS
    # ========================================================

    if transition == "AC":
        x_column = "K0"

    elif transition == "BCb":
        x_column = "Delta"

    else:
        raise ValueError(
            f"Unknown transition: {transition}"
        )

    # ========================================================
    # LOAD SUPERVISED
    # ========================================================

    supervised_raw = load_supervised_data(
        ROOT,
        transition,
        standardized=False,
    )

    supervised_std = load_supervised_data(
        ROOT,
        transition,
        standardized=True,
    )

    # ========================================================
    # LOAD UNSUPERVISED
    # ========================================================

    unsupervised_raw = load_unsupervised_data(
        ROOT,
        transition,
        standardized=False,
    )

    unsupervised_std = load_unsupervised_data(
        ROOT,
        transition,
        standardized=True,
    )

    # ========================================================
    # GET ALL MODELS
    # ========================================================

    models = get_unsupervised_models(
        unsupervised_raw,
        unsupervised_std,
    )

    print("=" * 70)
    print(f"TRANSITION: {transition}")
    print("=" * 70)
    print("Unsupervised models:")
    print(models)
    print()

    # ========================================================
    # CREATE ONE FIGURE PER MODEL
    # ========================================================

    for model in models:

        print(
            f"Plotting: {transition} — {model}"
        )

        fig, axes = make_unsupervised_figure(
            transition=transition,
            model=model,
            supervised_raw=supervised_raw,
            supervised_std=supervised_std,
            unsupervised_raw=unsupervised_raw,
            unsupervised_std=unsupervised_std,
            x_column=x_column,
        )
        pdf.savefig(
            fig,
            bbox_inches="tight",
        )
        
        plt.close(fig)


# ============================================================
# RUN
# ============================================================

output_dir = ROOT / "figures_unsupervised_comparison"
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

pdf_path = output_dir / "unsupervised_comparison.pdf"

with PdfPages(pdf_path) as pdf:

    make_all_unsupervised_figures(
        "AC",
        pdf,
    )

    make_all_unsupervised_figures(
        "BCb",
        pdf,
    )

print(f"Saved: {pdf_path}")

TRANSITION: AC
Unsupervised models:
['Agglomerative', 'BayesianGMM', 'Birch', 'DBSCAN', 'GaussianMixture', 'KMeans', 'MeanShift', 'Spectral']

Plotting: AC — Agglomerative
Plotting: AC — BayesianGMM
Plotting: AC — Birch
Plotting: AC — DBSCAN
Plotting: AC — GaussianMixture
Plotting: AC — KMeans
Plotting: AC — MeanShift
Plotting: AC — Spectral
TRANSITION: BCb
Unsupervised models:
['Agglomerative', 'BayesianGMM', 'Birch', 'DBSCAN', 'GaussianMixture', 'KMeans', 'MeanShift', 'Spectral']

Plotting: BCb — Agglomerative
Plotting: BCb — BayesianGMM
Plotting: BCb — Birch
Plotting: BCb — DBSCAN
Plotting: BCb — GaussianMixture
Plotting: BCb — KMeans
Plotting: BCb — MeanShift
Plotting: BCb — Spectral
Saved: /home/mariuszoslaw/uni/masters/Results/figures_unsupervised_comparison/unsupervised_comparison.pdf
